# Chat Completions API

## OpenAI API client

To explore building effective AI agents, we can start with pure Python and LLM APIs. In particular, we can use the [Python SDK](https://github.com/openai/openai-python/tree/main) for the [OpenAI API](https://platform.openai.com/docs/api-reference/introduction). First, we need to load the API key in the environmental variables. The client expects the environmental variable `OPENAI_API_KEY` which we can load from the `.env` file. This is easy enough to implement:

In [1]:
import inspect
from notebooks.utils import load_dotenv, print
print(inspect.getsource(load_dotenv))

def load_dotenv(verbose=False):
    with open(".env") as f:
        for line in f.readlines():
            k, v = line.split("=")
            os.environ[k] = v.strip().strip('"')
            if verbose:
                print(f"Loaded env variable: {k}")



In [2]:
load_dotenv(verbose=True)

Loaded env variable: OPENAI_API_KEY
Loaded env variable: GROQ_API_KEY


Then the API key is automatically read by the **client**:

In [3]:
from openai import OpenAI

client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "system", 
            "content": "You're a helpful assistant."},
        {
            "role": "user",
            "content": "Tell me about the history of GPT in a single paragraph.",
        },
    ],
)

response_text = completion.choices[0].message.content

:::{.callout-note}
We can have **multiple completions** of the same prompt. This allows choosing between the responses.
For example, we can set `temperature=0.9` to get more varied outputs, so that choosing becomes nontrivial. 
By default only 1 completion by default, hence `[0]`.
:::

In [4]:
print(response_text, wrap=True)

Generative Pre-trained Transformer (GPT) is a series of language models
developed by OpenAI, with its beginnings in 2018 with the release of GPT-1. The
series uses a transformer architecture and focuses on pre-training on a diverse
dataset followed by fine-tuning on specific tasks, allowing for impressive
capabilities in natural language understanding and generation. GPT-2, released
in 2019, gained attention for its improved fluency and size, but also raised
concerns about misuse, leading to a staged release. In 2020, GPT-3 significantly
advanced the model’s capabilities with 175 billion parameters, becoming widely
known for its ability to perform various language tasks with few examples. Each
iteration has improved the model's scale, training methods, and ability to
generate human-like text, influencing numerous applications in AI.


Entire model response (this includes **token usage** -- very important):

In [5]:
from pprint import pprint
pprint(completion.model_dump())

{'choices': [{'finish_reason': 'stop',
              'index': 0,
              'logprobs': None,
              'message': {'annotations': [],
                          'audio': None,
                          'content': 'Generative Pre-trained Transformer (GPT) '
                                     'is a series of language models developed '
                                     'by OpenAI, with its beginnings in 2018 '
                                     'with the release of GPT-1. The series '
                                     'uses a transformer architecture and '
                                     'focuses on pre-training on a diverse '
                                     'dataset followed by fine-tuning on '
                                     'specific tasks, allowing for impressive '
                                     'capabilities in natural language '
                                     'understanding and generation. GPT-2, '
                                     're

:::{.callout-note}
We are using the **chat completions** API where an autoregressive process that's running under the hood. Here the prompt to be completed is:

```python
messages=[
    {"role": "system", "content": "You are a poetic but terse assistant."},   # prompt
    {"role": "user", "content": "What is the color of the sky?"}              # prompt
]
```

And the completion is given by the API's output:

```python
{
  "role": "assistant",
  "content": "The sky shifts from azure to amber, a canvas for sun's daily journey."
}
```

**Remark.** From theory, the generated response is statistically the most likely continuation of the prompt text sequence. 
:::

## Structured outputs

[Structured outputs](https://platform.openai.com/docs/guides/structured-outputs?api-mode=responses) is a feature that ensures that a model generates responses that adhere to a supplied **schema** (e.g. a Pydantic model). As such, the output can then be parsed using the same Pydantic model. Structured outputs makes prompting significantly simpler: no more need for strongly worded prompts to achieve consistent formatting, no explicitly having to retry incorrectly formatted responses, or having invalid hallucinated values (can specify **enums**).

In [6]:
# https://platform.openai.com/docs/guides/structured-outputs?api-mode=responses
from openai import OpenAI
from pydantic import BaseModel

client = OpenAI()

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

completion = client.chat.completions.parse(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    response_format=CalendarEvent,
)

event = completion.choices[0].message.parsed
event

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

## Function calling

Also known as **tool calling**. Function calling give models access to external tools and data that they can use to respond to prompts. Since LLMs *only* consume and generate text, they cannot actually execute functions. Instead, the main program listens to the LLM hallucinate and executes the commands based on that (@fig-brainvat).

![**LLM as brain in a vat.** The LLM as core reasoning module thinking of what function to execute with what arguments without having the capability to execute them. We provide a separate process and environment for running the code. [Source](https://en.wikipedia.org/wiki/Brain_in_a_vat) ](./img/brain-vat.png){#fig-brainvat width=70%}

**Task.** To demonstrate tool calling, we develop a system of querying the weather in [Quisao](https://www.philatlas.com/luzon/r04a/rizal/pililla/quisao.html) using natural language. It builds on the OpenAI **chat completions API** (see @fig-chat-completions) where we incrementally append messages for the LLM. Our goal is to have the agent call the following API:

In [7]:
import json
import requests

def get_weather(latitude, longitude):
    """
    Get current weather data for provided coordinates with units:
    temperature (celsius), wind speed (kph), & precipitation (mm).
    """
    response = requests.get((
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&"
        "current=temperature_2m,wind_speed_10m,relative_humidity_2m,precipitation,precipitation_probability"
    ))
    data = response.json()
    return data["current"]


get_weather(latitude=14.4779, longitude=121.3214)  # true coordinates

{'time': '2025-09-16T01:00',
 'interval': 900,
 'temperature_2m': 28.4,
 'wind_speed_10m': 8.2,
 'relative_humidity_2m': 81,
 'precipitation': 0.0,
 'precipitation_probability': 20}

The entire implementation will look as follows:

![Creating a weather report using the chat completions API with LLM agents.](./img/chat-completions-api.png){#fig-chat-completions}

**Tool definition.** For the LLM to understand a specific tool, we have to define a schema that informs the model of what the tool does and its expected (required and optional) arguments. The following is the function definition for `get_weather`:

In [8]:
tools = [
    {
        "type": "function", # <1>
        "function": {
            "name": "get_weather",  # <2>
            "description": "Get current weather data for provided coordinates with units: temperature (celsius), wind speed (kph), & precipitation (mm).",
            "parameters": {
                "type": "object", # <3>
                "properties": {
                    "latitude": {
                        "type": "number"
                    },
                    "longitude": {
                        "type": "number"
                    },
                },
                "required": ["latitude", "longitude"],
                "additionalProperties": False,  # <4>
            },
            "strict": True, # <5>
        },
    }
]

1. Should always be function.
2. Function's name (i.e. `get_weather`).
3. Parameters are naturally JSON objects.
4. Part of JSON schema that determines whether extra fields are valid or not.
5. Not part of JSON schema, but OpenAI function calling option that *guides* the model to strictly follow the schema (i.e. not improvise). Setting `additionalProperties` to `False` and `strict` to `True` works together to ensure that the model generates the correct parameters schema.

:::{.callout-tip} 
Because `parameters` is defined by a JSON schema, you can leverage many of its rich features like property types, enums, descriptions, nested objects, and so on. For example, we can have:

```
"properties": {
    "unit": {
        "type": "string",
        "enum": ["celsius", "fahrenheit"],
        "description": "Unit of measure for temperature."
    }
}
```
:::

**Agent definition.** We define messages outside of the agent to track chat history.

In [9]:
system_prompt = "You are a helpful weather assistant."
messages = [{"role": "system", "content": system_prompt}]

def get_weather_agent(messages: list, update=True):
    """Agent with access to `get_weather` API. Responds with tool calls."""
    
    completion = client.chat.completions.create(
        model="gpt-4.1",
        messages=messages,
        tools=tools,
    )

    out = completion.choices[0].message
    if update:
        messages.append(out.model_dump())

    return out


# query step. expects tool_calls
query = "What's the weather like in Quisao, Pililla, Rizal right now?"
messages.append({"role": "user", "content": query})
response = get_weather_agent(messages)

pprint(response.model_dump())

{'annotations': [],
 'audio': None,
 'content': None,
 'function_call': None,
 'refusal': None,
 'role': 'assistant',
 'tool_calls': [{'function': {'arguments': '{"latitude":14.4864,"longitude":121.3407}',
                              'name': 'get_weather'},
                 'id': 'call_hkicnn05uZkveCL9avNyE0oQ',
                 'type': 'function'}]}


It's impressive that fairly accurate coordinates were obtained without using web search (i.e. these facts were inferred from the model weights). Observe that whenever we have `tools`, the model responds with only `tool_calls` as nonnull (e.g. `content` is empty) with role `assistant`.  

**Function calls.** We will iterate over tool calls and process them separately. Each function call and their results are then logged in the message history. This explains why we defined `messages` outside of the API call unlike the usual setup .

In [10]:
def call_function(name, args):
    fn = {
        "get_weather": get_weather  
    }
    return fn[name](**args)


for tool_call in response.tool_calls:
    args = json.loads(tool_call.function.arguments)
    name = tool_call.function.name
    tool_output = call_function(name, args)
    
    messages.append({    
        "role": "tool",     # <1>
        "tool_call_id": tool_call.id, 
        "content": json.dumps(tool_output)
    })

1. Tool calls are logged with role `tool`. 

Message history appended with actual API output:

In [11]:
pprint(messages)

[{'content': 'You are a helpful weather assistant.', 'role': 'system'},
 {'content': "What's the weather like in Quisao, Pililla, Rizal right now?",
  'role': 'user'},
 {'annotations': [],
  'audio': None,
  'content': None,
  'function_call': None,
  'refusal': None,
  'role': 'assistant',
  'tool_calls': [{'function': {'arguments': '{"latitude":14.4864,"longitude":121.3407}',
                               'name': 'get_weather'},
                  'id': 'call_hkicnn05uZkveCL9avNyE0oQ',
                  'type': 'function'}]},
 {'content': '{"time": "2025-09-16T01:00", "interval": 900, "temperature_2m": '
             '27.7, "wind_speed_10m": 8.2, "relative_humidity_2m": 81, '
             '"precipitation": 0.0, "precipitation_probability": 0}',
  'role': 'tool',
  'tool_call_id': 'call_hkicnn05uZkveCL9avNyE0oQ'}]


:::{.callout-note}
Chat completion API calls have stateless single request-response cycles which gives the user straightforward control over the **message history**. This can be seen in the above example where we manually manage message history with tool calls declaration as well as actual function outputs.
:::

**Report agent.** Next, we pass this thread to another agent (possibly to a different, more specialized model) which will process the outputs of the tool calls along with earlier chat messages. To take advantage of structured outputs we again define a response format. Here we use Pydantic `Field` with a description that helps the LLM.

In [12]:
from pydantic import Field

class WeatherReport(BaseModel):
    response: str = Field(description="A natural language response to the user's question.")
    temperature: float = Field(description="Current temperature in celsius for the given location.")


def weather_report_agent(messages: list, update=True):
    completion = client.chat.completions.parse(
        model="gpt-4o",
        messages=messages,
        tools=tools,    # <!>
        response_format=WeatherReport,
    )
    
    out = completion.choices[0].message
    if update:
        messages.append(out.model_dump())
    
    return out

:::{.callout-caution}
The aggregator also needs access to tools for it to understand the context of each tool call!
:::

**Final output.** Note that the units correctly identified from the `get_weather` docstring:

In [13]:
response = weather_report_agent(messages)
weather_report = response.parsed
print("temp:", weather_report.temperature, "\n")
print(weather_report.response, wrap=True)

temp: 27.7 

The current weather in Quisao, Pililla, Rizal is quite warm with a temperature
of 27.7°C. The wind is blowing at a speed of 8.2 kph, and there's no
precipitation at the moment, making it a dry period. The humidity is relatively
high at 81%, so it might feel a bit muggy.


## Appendix: Utility functions

### Chat completion

The following helper functions are saved in the `notebooks.agents` library. First, the usual boilerplate when dealing with the chat completions API.

In [14]:
from notebooks.utils import display_python
from notebooks.agents.chat import *

In [15]:
#| echo: false
from openai import OpenAI
from pydantic import BaseModel
from notebooks.utils import load_dotenv
load_dotenv()
client = OpenAI()

Next, we abstract over the chat completions API:

In [16]:
display_python(ChatCompletions)

This allows setting a default model for the completions API:

In [17]:
class Greeting(BaseModel):
    greeting: str

MODEL = "gpt-5-nano"
completions = ChatCompletions(client, default_model=MODEL)
messages = [message_dict(role="user", prompt="hi chat")]

print(completions.create(messages=messages))
print(completions.parsed(messages=messages, schema=Greeting))

Hi there! How can I help today? I can explain concepts, answer questions, help with writing or coding, brainstorm ideas, translate, or just chat. Tell me what you’d like to do or ask a specific question.
{'greeting': 'Hi there! How can I help you today?'}


:::{.callout-note}
The structured output schema influences the generation of the model.
:::

### Chat history

Here we define the `Role` class which is always better than typing roles out.

In [18]:
display_python(Role)

In [19]:
display_python(message_dict)

In [20]:
# example
print(Role.get_valid_roles())

try:
    print(message_dict(role="user", prompt="Hello", tag="greeting"))
    print(message_dict(role="test", prompt="Hello", tag="greeting"))
except ValueError as e:
    print(e)

{'user', 'system', 'assistant', 'tool'}
{'role': 'user', 'content': '<greeting>Hello</greeting>'}
Invalid role: test


For the chat history class, we implement a limit parameter `max_len` to naively prevent [context overflow](https://aws.amazon.com/blogs/security/context-window-overflow-breaking-the-barrier/) by dropping the oldest messages. The parameter `fixed_n` (default `1`) is used to preserve first `n` message since it is often important (e.g. `n=1` for the system prompt).

In [21]:
display_python(ChatHistory)

In [22]:
chat_history = ChatHistory(
    system_prompt="you are a goldfish", max_len=3, fixed_n=1
)
chat_history.update("1", "user")
chat_history.update("2", "user")
chat_history.update("3", "user")
chat_history

[{'role': 'system', 'content': 'you are a goldfish'},
 {'role': 'user', 'content': '2'},
 {'role': 'user', 'content': '3'}]

Only valid roles are allowed:

In [23]:
for cmd in [
    lambda: chat_history.append({"role": "test", "content": "test"}),
    lambda: chat_history.update(role="test", prompt="test")
]:
    try:
        cmd()
    except ValueError as e:
        print(e)

Invalid role: test
Invalid role: test


### Tag extraction

The following utilities are for extracting content from tags (e.g. `<thought>`, `<response>`, etc).

In [24]:
from notebooks.agents.utils import *

In [25]:
#| echo: false
display_python(TagContentResult)
print()
display_python(extract_tag_content)

Adding tags:

In [26]:
chat = ChatHistory()
chat.update("This is a thought.",  role="user", tag="thought")
chat

[{'role': 'user', 'content': '<thought>This is a thought.</thought>'}]

Extracting tag content:

In [27]:
messages = """
<thought>This is a thought.</thought>
<response>This is a response.</response>
<tool>This is a tool-use.</tool> 
"""

print(extract_tag_content(messages, "thought"))
print(extract_tag_content(messages, "response"))
print(extract_tag_content(messages, "tool"))

TagContentResult(content=['This is a thought.'], found=True)
TagContentResult(content=['This is a response.'], found=True)
TagContentResult(content=['This is a tool-use.'], found=True)
